# PART2 Colab GPU Runner

Run this notebook in Google Colab with `Runtime > Change runtime type > GPU`.

Outputs are stored in Google Drive under `/content/drive/MyDrive/NLP_Study_PART2` so checkpoints, predictions, results, and logs survive Colab disconnects. Colab runtime limits cannot be guaranteed away; the reliable protection is saving work to Drive.

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
BASE=/content/drive/MyDrive/NLP_Study_PART2
if [ ! -d "$BASE/.git" ]; then
  git clone -b part2 --single-branch https://github.com/Y0onSe0/NLP_Study.git "$BASE"
fi
cd "$BASE"
git fetch origin part2
git checkout part2
git pull --ff-only origin part2
mkdir -p checkpoints predictions results logs
git branch
git status --short

In [ ]:
%cd /content/drive/MyDrive/NLP_Study_PART2
!pip -q install transformers==4.46.3 tokenizers==0.20.0 einops==0.8.0 sacrebleu==2.5.1 explainaboard_client==0.0.7 tqdm==4.58.0

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU is not enabled. Use Runtime > Change runtime type > GPU.'
print(torch.cuda.get_device_name(0))

## 1. Smoke Test

Run this first. If it fails with CUDA memory issues, reduce `--batch_size` to `1`.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/NLP_Study_PART2
python paraphrase_detection.py \
  --use_gpu \
  --mode train_dev \
  --epochs 1 \
  --batch_size 2 \
  --max_train_examples 128 \
  --max_dev_examples 128 \
  --prompt_template baseline \
  --output_tag smoke-baseline 2>&1 | tee logs/smoke-baseline.log

## 2. Prompt Screening

This compares `baseline`, `direct`, and `meaning` on a subset. Pick the best prompt from `results/paraphrase_experiments.csv`.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/NLP_Study_PART2
for PROMPT in baseline direct meaning; do
  python paraphrase_detection.py \
    --use_gpu \
    --mode train_dev \
    --epochs 1 \
    --batch_size 8 \
    --max_train_examples 20000 \
    --max_dev_examples 5000 \
    --prompt_template "$PROMPT" \
    --output_tag "screen-$PROMPT" 2>&1 | tee "logs/screen-$PROMPT.log"
done
tail -n 10 results/paraphrase_experiments.csv

In [ ]:
import pandas as pd
path = '/content/drive/MyDrive/NLP_Study_PART2/results/paraphrase_experiments.csv'
df = pd.read_csv(path)
display(df.tail(10))
screen = df[df['output_tag'].astype(str).str.startswith('screen-')].copy()
screen['dev_acc'] = pd.to_numeric(screen['dev_acc'], errors='coerce')
display(screen.sort_values('dev_acc', ascending=False).head(5))

## 3. Full Training

Set `BEST_PROMPT` to the best prompt from screening. Valid values are `baseline`, `direct`, `meaning`.

The best checkpoint is saved to `checkpoints/prompt-full.pt` in Drive whenever dev accuracy improves.

In [ ]:
BEST_PROMPT = 'meaning'  # change after screening if needed: baseline, direct, meaning
DEV_SELECTED_THRESHOLD = '0.50'  # replace after threshold calibration
with open('/content/part2_run.env', 'w') as f:
    f.write(f'BEST_PROMPT={BEST_PROMPT}\n')
    f.write(f'DEV_SELECTED_THRESHOLD={DEV_SELECTED_THRESHOLD}\n')
print('BEST_PROMPT =', BEST_PROMPT)
print('DEV_SELECTED_THRESHOLD =', DEV_SELECTED_THRESHOLD)

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/NLP_Study_PART2
source /content/part2_run.env
python paraphrase_detection.py \
  --use_gpu \
  --mode train_dev \
  --epochs 10 \
  --batch_size 8 \
  --lr 1e-5 \
  --prompt_template "$BEST_PROMPT" \
  --output_tag prompt-full \
  --filepath checkpoints/prompt-full.pt 2>&1 | tee logs/prompt-full.log
test -f checkpoints/prompt-full.pt && echo checkpoint-ok

If Colab runs out of memory, lower `--batch_size` to `4` or `2`. If it is too slow, keep `gpt2` and do not switch to larger models.

## 4. Bidirectional Dev Prediction

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/NLP_Study_PART2
source /content/part2_run.env
python paraphrase_detection.py \
  --use_gpu \
  --mode dev_predict \
  --filepath checkpoints/prompt-full.pt \
  --prompt_template "$BEST_PROMPT" \
  --bidirectional \
  --threshold 0.5 \
  --output_tag prompt-bidir-dev 2>&1 | tee logs/prompt-bidir-dev.log

## 5. Threshold Calibration

Use the printed `best dev threshold` value in the next cells.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/NLP_Study_PART2
source /content/part2_run.env
python paraphrase_detection.py \
  --use_gpu \
  --mode calibrate_dev \
  --filepath checkpoints/prompt-full.pt \
  --prompt_template "$BEST_PROMPT" \
  --bidirectional \
  --threshold_min 0.30 \
  --threshold_max 0.70 \
  --threshold_step 0.01 \
  --output_tag prompt-bidir-calib 2>&1 | tee logs/prompt-bidir-calib.log

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/NLP_Study_PART2/results/paraphrase_experiments.csv')
display(df.tail(20))

## 6. Error Analysis

In [ ]:
BEST_PROMPT = 'meaning'  # keep this equal to the selected prompt
DEV_SELECTED_THRESHOLD = '0.50'  # replace with the calibrated threshold
with open('/content/part2_run.env', 'w') as f:
    f.write(f'BEST_PROMPT={BEST_PROMPT}\n')
    f.write(f'DEV_SELECTED_THRESHOLD={DEV_SELECTED_THRESHOLD}\n')
print(BEST_PROMPT, DEV_SELECTED_THRESHOLD)

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/NLP_Study_PART2
source /content/part2_run.env
python paraphrase_detection.py \
  --use_gpu \
  --mode error_analysis \
  --filepath checkpoints/prompt-full.pt \
  --prompt_template "$BEST_PROMPT" \
  --bidirectional \
  --threshold "$DEV_SELECTED_THRESHOLD" \
  --output_tag prompt-bidir-error 2>&1 | tee logs/prompt-bidir-error.log

## 7. Final Test Prediction

Run once after checkpoint, prompt, bidirectional setting, and threshold are fixed from dev.

In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/NLP_Study_PART2
source /content/part2_run.env
python paraphrase_detection.py \
  --use_gpu \
  --mode test_predict \
  --filepath checkpoints/prompt-full.pt \
  --prompt_template "$BEST_PROMPT" \
  --bidirectional \
  --threshold "$DEV_SELECTED_THRESHOLD" \
  --para_test_out predictions/para-test-final.csv \
  --output_tag final-test 2>&1 | tee logs/final-test.log
wc -l predictions/para-test-final.csv
head -n 5 predictions/para-test-final.csv
tail -n 5 results/paraphrase_experiments.csv
python prepare_submit.py || true